#### 1) Environment & GPU check

In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))


Torch version: 2.5.1+cu121
CUDA available: True
Device: cuda


#### 2) Imports (HF + SentenceTransformers)

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
import numpy as np
import pandas as pd

#### 3) Define models (classification + embeddings + summarisation)

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"  # placeholder for clinical classifier
EMB_MODEL_NAME = "all-MiniLM-L6-v2"
SUM_MODEL_NAME = "google/pegasus-xsum"  # or "facebook/bart-large-cnn"

tokenizer_cls = AutoTokenizer.from_pretrained(CLASS_MODEL_NAME)
model_cls = AutoModelForSequenceClassification.from_pretrained(CLASS_MODEL_NAME).to(DEVICE)
model_cls.eval()

emb_model = SentenceTransformer(EMB_MODEL_NAME, device=DEVICE)

sum_tokenizer = AutoTokenizer.from_pretrained(SUM_MODEL_NAME)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(SUM_MODEL_NAME).to(DEVICE)
sum_model.eval()


tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

C:\Users\marke\projects\data-science\nhs\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\marke\.cache\huggingface\hub\models--google--pegasus-xsum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

PegasusForConditionalGeneration(
  (model): PegasusModel(
    (shared): Embedding(96103, 1024, padding_idx=0)
    (encoder): PegasusEncoder(
      (embed_tokens): Embedding(96103, 1024, padding_idx=0)
      (embed_positions): PegasusSinusoidalPositionalEmbedding(512, 1024)
      (layers): ModuleList(
        (0-15): 16 x PegasusEncoderLayer(
          (self_attn): PegasusAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_layer_nor

#### 4) Sample NHS‑style clinical texts

In [4]:
clinical_notes = [
    "Patient presents with chest pain radiating to left arm, sweating, and shortness of breath. ECG shows ST elevation.",
    "Patient reports mild headache and fatigue. Observations within normal range. Discharged with advice to rest.",
    "Post‑operative patient with fever and elevated CRP. Wound site appears red and swollen. Possible infection.",
    "Elderly patient with history of COPD, increased breathlessness over 3 days, using inhaler more frequently.",
    "Patient attended A&E following a fall. X‑ray confirms fractured wrist. Plaster applied, follow‑up in fracture clinic."
]

pd.DataFrame({"note": clinical_notes})


,note
0,Patient presents with chest pain radiating to ...
1,Patient reports mild headache and fatigue. Obs...
2,Post‑operative patient with fever and elevated...
3,"Elderly patient with history of COPD, increase..."
4,Patient attended A&E following a fall. X‑ray c...


#### 5) simple cleaning

In [12]:
import re

def clean_text(text: str) -> str:
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

cleaned_notes = [clean_text(t) for t in clinical_notes]
pd.DataFrame({"note": cleaned_notes})


,note
0,Patient presents with chest pain radiating to ...
1,Patient reports mild headache and fatigue. Obs...
2,Post‑operative patient with fever and elevated...
3,"Elderly patient with history of COPD, increase..."
4,Patient attended A&E following a fall. X‑ray c...


#### classification

In [13]:
def classify_texts(texts):
    inputs = tokenizer_cls(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model_cls(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

    return probs

probs = classify_texts(cleaned_notes)
probs_df = pd.DataFrame(probs, columns=["negative", "positive"])
pd.concat([pd.DataFrame({"note": cleaned_notes}), probs_df], axis=1)


,note,negative,positive
0,Patient presents with chest pain radiating to ...,0.988716,0.011284
1,Patient reports mild headache and fatigue. Obs...,0.995276,0.004724
2,Post‑operative patient with fever and elevated...,0.995920,0.004080
3,"Elderly patient with history of COPD, increase...",0.936088,0.063912
4,Patient attended A&E following a fall. X‑ray c...,0.886202,0.113798


#### Embeddings for retrieval / similarity

In [14]:
embeddings = emb_model.encode(cleaned_notes, convert_to_tensor=True)
embeddings.shape

torch.Size([5, 384])

#### retrieve most similar notes for a query

In [15]:
def retrieve_similar(query: str, top_k: int = 3):
    q_emb = emb_model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, embeddings)[0]
    top_results = torch.topk(scores, k=top_k)

    results = []
    for idx, score in zip(top_results.indices, top_results.values):
        results.append({
            "note": cleaned_notes[idx],
            "score": float(score)
        })
    return pd.DataFrame(results)

query = "Possible infection after surgery"
retrieve_similar(query, top_k=3)


,note,score
0,Post‑operative patient with fever and elevated...,0.594624
1,Patient attended A&E following a fall. X‑ray c...,0.251193
2,Patient presents with chest pain radiating to ...,0.244488


#### Summarisation of long clinical text (claims / reports)

In [16]:
def summarise(text: str, max_length: int = 64):
    inputs = sum_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="longest"
    ).to(DEVICE)

    with torch.no_grad():
        summary_ids = sum_model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            early_stopping=True
        )

    summary = sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

long_note = (
    "Patient presents with chest pain radiating to left arm, sweating, and shortness of breath. "
    "ECG shows ST elevation consistent with myocardial infarction. "
    "Aspirin and GTN administered in A&E, transferred to cardiology for further management."
)

print("Original:\n", long_note)
print("\nSummary:\n", summarise(long_note))


Original:
 Patient presents with chest pain radiating to left arm, sweating, and shortness of breath. ECG shows ST elevation consistent with myocardial infarction. Aspirin and GTN administered in A&E, transferred to cardiology for further management.

Summary:
 We report the case of a 65-year-old man with a history of heart disease who presented with chest pain.


#### end to end pipeline function

In [17]:
def clinical_pipeline(note: str, query_for_similarity: str = None):
    cleaned = clean_text(note)

    # Classification
    cls_probs = classify_texts([cleaned])[0]
    severity_proxy = "high" if cls_probs[1] > 0.7 else "low/medium"

    # Embedding
    emb = emb_model.encode(cleaned, convert_to_tensor=True)

    # Summarisation
    summary = summarise(cleaned)

    # Similarity (optional)
    similar_df = None
    if query_for_similarity is not None:
        similar_df = retrieve_similar(query_for_similarity, top_k=3)

    return {
        "cleaned": cleaned,
        "severity_proxy": severity_proxy,
        "classification_probs": cls_probs,
        "summary": summary,
        "similar": similar_df
    }

example_note = clinical_notes[2]
result = clinical_pipeline(example_note, query_for_similarity="infection after surgery")

print("NOTE:\n", result["cleaned"])
print("\nSEVERITY (proxy):", result["severity_proxy"])
print("\nSUMMARY:\n", result["summary"])
print("\nSIMILAR NOTES:\n", result["similar"])


NOTE:
 Post‑operative patient with fever and elevated CRP. Wound site appears red and swollen. Possible infection.

SEVERITY (proxy): low/medium

SUMMARY:
 A patient has undergone surgery for a perforated ulcer.

SIMILAR NOTES:
                                                 note     score
0  Post‑operative patient with fever and elevated...  0.558132
1  Patient attended A&E following a fall. X‑ray c...  0.251775
2  Patient presents with chest pain radiating to ...  0.191833


## BEST MODELS TO RUN ON A LAPTOP (CLINICAL NLP)

These models are chosen because they:
- fit comfortably in 4–8 GB VRAM
- run fast on consumer GPUs (RTX 3050/3060/3070/4050/4060)
- work well on CPU if needed
- are ideal for NHS Resolution use cases (claims, incidents, harm themes, severity)

------------------------------------------------------------
1. BEST CLINICAL EMBEDDING MODEL (LAPTOP)
------------------------------------------------------------
MiniLM-L6-v2
- 33M parameters
- Extremely fast
- Excellent semantic embeddings
- Perfect for retrieval, clustering, similarity search
- Works on CPU or GPU
- Ideal for FAISS-based RAG

Use for:
- Similar claims
- Similar incidents
- Harm theme clustering
- Semantic search

------------------------------------------------------------
2. BEST CLINICAL CLASSIFICATION MODEL (LAPTOP)
------------------------------------------------------------
Bio_ClinicalBERT
- 110M parameters
- Trained on MIMIC-III clinical notes
- Understands clinical language far better than general BERT
- Runs easily on laptop GPUs

Use for:
- Harm classification
- Severity prediction
- Incident type classification
- Clinical triage

------------------------------------------------------------
3. BEST SUMMARISATION MODEL (LAPTOP)
------------------------------------------------------------
PEGASUS-XSum
- Best open-source abstractive summariser
- Works on laptop GPU
- Slow but manageable

Alternative:
BART-Large-CNN
- More stable
- Slightly slower
- Good for long claim documents

Use for:
- Summarising long claim narratives
- Summarising incident reports
- Extracting key events

------------------------------------------------------------
4. BEST RAG RETRIEVAL MODEL (LAPTOP)
------------------------------------------------------------
E5-Base
- Very strong retrieval performance
- Works well on laptop GPUs
- Perfect for FAISS indexing

Use for:
- RAG over claims database
- Retrieval of similar incidents
- Linking new claims to historical precedents

------------------------------------------------------------
5. BEST GENERAL EMBEDDING MODEL (LAPTOP)
------------------------------------------------------------
MPNet-Base
- Excellent semantic embeddings
- Good for clustering and similarity
- Works on laptop GPUs

Use for:
- Harm theme clustering
- Semantic grouping of incidents
- General-purpose embeddings

------------------------------------------------------------
6. BEST MEDICAL TERMINOLOGY MODEL (OPTIONAL)
------------------------------------------------------------
PubMedBERT
- Trained on PubMed abstracts + full text
- Strong for medical terminology
- Runs on laptop GPUs

Use for:
- Diagnosis-related text
- Medical terminology extraction
- Clinical concept matching

------------------------------------------------------------
RECOMMENDED STACK FOR A LAPTOP (SHORTLIST)
------------------------------------------------------------
If you want the best possible clinical NLP pipeline on a laptop:

1. Clinical classification:
   Bio_ClinicalBERT

2. Embeddings (semantic search, clustering):
   MiniLM-L6-v2

3. Retrieval (RAG):
   E5-Base

4. Summarisation:
   PEGASUS-XSum

5. Optional medical terminology:
   PubMedBERT

This combination is:
- fully open-source
- clinically relevant
- laptop-friendly
- ideal for NHS Resolution use cases
- perfect for notebooks and demos


If you want, I can now generate:

a model loader notebook using these models

a clinical NLP pipeline notebook using this exact stack

a full RAG notebook using MiniLM/E5 + FAISS

a fine‑tuning notebook using Bio_ClinicalBERT